# 🚖 RideSense-AI: NYC Taxi Trip Duration Prediction

## Notebook 01: Data Understanding

---

### Project Overview

RideSense-AI is an end-to-end Machine Learning project that aims to predict the duration of taxi trips in New York City using historical trip data.

This notebook focuses on understanding the dataset before performing any preprocessing or exploratory data analysis.

A thorough understanding of the dataset helps identify data quality issues, understand feature types, and prepare an effective strategy for the subsequent stages of the machine learning pipeline.

# Problem Statement

The objective of this project is to predict the total duration of a taxi trip in New York City.

Accurate trip duration prediction can help:

- Improve fare estimation
- Optimize driver allocation
- Enhance customer experience
- Support route planning
- Reduce waiting time

The prediction will be made using information available before the trip starts, such as pickup location, dropoff location, passenger count, pickup time, and vendor information.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Loading the Dataset

The NYC Taxi Trip Duration dataset consists of two files:

- **Training Dataset** – Contains historical taxi trip records along with the target variable (`trip_duration`).
- **Test Dataset** – Contains trip information without the target variable and is used for prediction.

In [2]:
train_path = "../data/raw/train.csv"
test_path = "../data/raw/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Dataset Preview

Let's inspect the first few records from both datasets to understand their structure.

In [3]:
train_df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [20]:
test_df.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N


## Observation

From the dataset preview, we observe that:

- The **training dataset** contains **11 columns**.
- The **test dataset** contains **9 columns**.

The following columns are present only in the training dataset:

- `dropoff_datetime`
- `trip_duration`

This indicates that the training dataset contains complete trip information along with the target variable, whereas the test dataset contains only the input features required for making predictions.

# Dataset Dimensions

In [21]:
print("Train Dataset Shape :", train_df.shape)
print("Test Dataset Shape :", test_df.shape)

Train Dataset Shape : (1458644, 11)
Test Dataset Shape : (625134, 9)


## Observation

- Training Dataset contains **1,458,644** rows and **11** columns.
- Test Dataset contains **625,134** rows and **9** columns.

# Dataset Information

Let's inspect the data types and memory usage of both datasets.

In [22]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1458644 non-null  object 
 1   vendor_id           1458644 non-null  int64  
 2   pickup_datetime     1458644 non-null  object 
 3   dropoff_datetime    1458644 non-null  object 
 4   passenger_count     1458644 non-null  int64  
 5   pickup_longitude    1458644 non-null  float64
 6   pickup_latitude     1458644 non-null  float64
 7   dropoff_longitude   1458644 non-null  float64
 8   dropoff_latitude    1458644 non-null  float64
 9   store_and_fwd_flag  1458644 non-null  object 
 10  trip_duration       1458644 non-null  int64  
dtypes: float64(4), int64(3), object(4)
memory usage: 122.4+ MB


In [23]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625134 entries, 0 to 625133
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  625134 non-null  object 
 1   vendor_id           625134 non-null  int64  
 2   pickup_datetime     625134 non-null  object 
 3   passenger_count     625134 non-null  int64  
 4   pickup_longitude    625134 non-null  float64
 5   pickup_latitude     625134 non-null  float64
 6   dropoff_longitude   625134 non-null  float64
 7   dropoff_latitude    625134 non-null  float64
 8   store_and_fwd_flag  625134 non-null  object 
dtypes: float64(4), int64(2), object(3)
memory usage: 42.9+ MB


## Observation

The dataset contains:

### Numerical Features

- vendor_id
- passenger_count
- pickup_longitude
- pickup_latitude
- dropoff_longitude
- dropoff_latitude
- trip_duration

### Categorical Features

- id
- store_and_fwd_flag

### Datetime Features (Currently Object)

- pickup_datetime
- dropoff_datetime

The datetime columns will later be converted into proper datetime format.

# Dataset Dictionary

In [24]:
feature_summary = pd.DataFrame({
    "Feature": train_df.columns,
    "Data Type": train_df.dtypes.values,
    "Description": [
        "Unique trip identifier",
        "Taxi vendor identifier",
        "Pickup date and time",
        "Dropoff date and time",
        "Number of passengers",
        "Pickup longitude",
        "Pickup latitude",
        "Dropoff longitude",
        "Dropoff latitude",
        "Store and Forward Flag",
        "Trip duration in seconds (Target Variable)"
    ]
})

feature_summary

,Feature,Data Type,Description
0,id,object,Unique trip identifier
1,vendor_id,int64,Taxi vendor identifier
2,pickup_datetime,object,Pickup date and time
3,dropoff_datetime,object,Dropoff date and time
4,passenger_count,int64,Number of passengers
5,pickup_longitude,float64,Pickup longitude
6,pickup_latitude,float64,Pickup latitude
7,dropoff_longitude,float64,Dropoff longitude
8,dropoff_latitude,float64,Dropoff latitude
9,store_and_fwd_flag,object,Store and Forward Flag


# Missing Values Analysis

In [25]:
train_df.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [26]:
test_df.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
dtype: int64

## Observation

No missing values were found in either the training or test dataset.

This indicates that no imputation techniques are required during preprocessing.

# Duplicate Records

In [27]:
print("Train Duplicates :", train_df.duplicated().sum())
print("Test Duplicates :", test_df.duplicated().sum())

Train Duplicates : 0
Test Duplicates : 0


## Observation

No duplicate records were found in either dataset.

# Statistical Summary

In [28]:
train_df.describe()

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
count,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06
mean,1.534950e+00,1.664530e+00,-7.397349e+01,4.075092e+01,-7.397342e+01,4.075180e+01,9.594923e+02
std,4.987772e-01,1.314242e+00,7.090186e-02,3.288119e-02,7.064327e-02,3.589056e-02,5.237432e+03
min,1.000000e+00,0.000000e+00,-1.219333e+02,3.435970e+01,-1.219333e+02,3.218114e+01,1.000000e+00
25%,1.000000e+00,1.000000e+00,-7.399187e+01,4.073735e+01,-7.399133e+01,4.073588e+01,3.970000e+02
50%,2.000000e+00,1.000000e+00,-7.398174e+01,4.075410e+01,-7.397975e+01,4.075452e+01,6.620000e+02
75%,2.000000e+00,2.000000e+00,-7.396733e+01,4.076836e+01,-7.396301e+01,4.076981e+01,1.075000e+03
max,2.000000e+00,9.000000e+00,-6.133553e+01,5.188108e+01,-6.133553e+01,4.392103e+01,3.526282e+06


## Initial Observations

From the statistical summary, we observe:

- Passenger count ranges from **0** to **9**, where a value of **0** appears to be invalid.
- Trip duration ranges from **1 second** to **3,526,282 seconds**, indicating the presence of extreme outliers.
- Pickup and dropoff coordinates cover a wider geographical range than expected for New York City, suggesting GPS anomalies.

# Target Variable

In [29]:
print("Minimum Trip Duration :", train_df["trip_duration"].min(), "seconds")
print("Maximum Trip Duration :", train_df["trip_duration"].max(), "seconds")
print("Average Trip Duration :", round(train_df["trip_duration"].mean(),2), "seconds")

Minimum Trip Duration : 1 seconds
Maximum Trip Duration : 3526282 seconds
Average Trip Duration : 959.49 seconds


## Observation

The target variable (`trip_duration`) is measured in **seconds**.

The presence of extremely large values suggests that outlier analysis will be an important step before model training.

# Memory Usage

In [30]:
memory_usage = train_df.memory_usage(deep=True).sum() / 1024**2

print(f"Total Memory Usage : {memory_usage:.2f} MB")

Total Memory Usage : 417.32 MB


# Overall Data Quality Assessment

### Positive Findings

- No missing values
- No duplicate records
- Large dataset suitable for machine learning
- Rich combination of temporal, categorical, and geographical features

### Potential Issues

- Passenger count contains unrealistic values.
- Trip duration contains extreme outliers.
- Datetime columns require conversion.
- GPS coordinates require validation.

# Conclusion

In this notebook, we explored and understood the structure of the NYC Taxi Trip Duration dataset.

### Key Findings

- Training dataset contains **1.45 million** taxi trips.
- Test dataset contains **625 thousand** trips.
- No missing values were detected.
- No duplicate records were found.
- Datetime columns require conversion.
- Passenger count, trip duration, and geographical coordinates require further investigation.

### Next Step

The next notebook will focus on **Exploratory Data Analysis (EDA)**, where we will analyze feature distributions, identify outliers, study temporal and geographical patterns, and understand the relationships between variables before preprocessing and feature engineering.

In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/train_featured.csv")

print(df.columns.tolist())
print("Total Columns:", len(df.columns))

['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_weekday', 'is_weekend', 'is_rush_hour', 'is_night', 'haversine_distance_km', 'manhattan_distance_km', 'longitude_difference', 'latitude_difference', 'bearing']
Total Columns: 20
